# Feature Engineering Guide for Stock Prediction
## ML for Engineers - Module 3: Regression

This notebook demonstrates how to use the `feature_engineering.py` library to create powerful features for stock price prediction.

**What you'll learn:**
- Load and prepare stock data
- Calculate technical indicators (RSI, MACD, Bollinger Bands, etc.)
- Visualize features
- Prepare data for machine learning

**Prerequisites:**
- Basic Python and pandas knowledge
- Understanding of stock data (OHLCV)

Created: 2026-01-05

## 1. Setup and Import Libraries

In [ ]:
# Install required libraries (run once)
!pip install yfinance pandas numpy matplotlib seaborn --quiet

print("✅ Libraries installed!")

In [ ]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import our feature engineering library
import feature_engineering as fe

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✅ Libraries imported successfully!")

## 2. Load Stock Data

Let's start by loading historical stock data. We'll use Apple (AAPL) as an example.

In [ ]:
# Choose your stock ticker
ticker = "AAPL"  # Change this to any stock you want!

# Download 5 years of data
print(f"Downloading {ticker} data...")
data = yf.download(ticker, start="2019-01-01", end="2024-01-01")

print(f"\n✅ Downloaded {len(data)} days of data")
print(f"   Date range: {data.index[0]} to {data.index[-1]}")

# Display first few rows
data.head(10)

In [ ]:
# Visualize closing price
plt.figure(figsize=(15, 6))
plt.plot(data.index, data['Close'], linewidth=2, label='Closing Price')
plt.title(f'{ticker} Stock Price (2019-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Current price: ${data['Close'].iloc[-1]:.2f}")
print(f"Price change: ${data['Close'].iloc[-1] - data['Close'].iloc[0]:.2f}")
print(f"Percentage gain: {((data['Close'].iloc[-1] / data['Close'].iloc[0]) - 1) * 100:.2f}%")

## 3. Feature Engineering: Individual Indicators

Let's calculate technical indicators one by one to understand what each does.

### 3.1 Moving Averages (MA)

Moving averages smooth out price data to identify trends.

In [ ]:
# Calculate moving averages
data['MA_10'] = fe.calculate_moving_average(data, window=10)
data['MA_50'] = fe.calculate_moving_average(data, window=50)
data['MA_200'] = fe.calculate_moving_average(data, window=200)

print("✅ Moving averages calculated!")

# Visualize
plt.figure(figsize=(15, 6))
plt.plot(data.index, data['Close'], linewidth=2, label='Close Price', alpha=0.7)
plt.plot(data.index, data['MA_10'], linewidth=2, label='MA 10-day', alpha=0.8)
plt.plot(data.index, data['MA_50'], linewidth=2, label='MA 50-day', alpha=0.8)
plt.plot(data.index, data['MA_200'], linewidth=2, label='MA 200-day', alpha=0.8)
plt.title(f'{ticker} with Moving Averages', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("   - Short MA (10-day) follows price closely")
print("   - Long MA (200-day) shows overall trend")
print("   - When short MA crosses above long MA = Bullish signal (Golden Cross)")
print("   - When short MA crosses below long MA = Bearish signal (Death Cross)")

### 3.2 Relative Strength Index (RSI)

RSI measures momentum and identifies overbought/oversold conditions.

In [ ]:
# Calculate RSI
data['RSI'] = fe.calculate_rsi(data, window=14)

print("✅ RSI calculated!")

# Visualize
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Price chart
ax1.plot(data.index, data['Close'], linewidth=2, label='Close Price')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title(f'{ticker} Price and RSI', fontsize=16, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# RSI chart
ax2.plot(data.index, data['RSI'], linewidth=2, label='RSI', color='purple')
ax2.axhline(y=70, color='r', linestyle='--', linewidth=2, label='Overbought (70)')
ax2.axhline(y=30, color='g', linestyle='--', linewidth=2, label='Oversold (30)')
ax2.fill_between(data.index, 30, 70, alpha=0.1, color='gray')
ax2.set_ylabel('RSI', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("   - RSI > 70: Stock is overbought (might go down soon)")
print("   - RSI < 30: Stock is oversold (might go up soon)")
print("   - RSI around 50: Neutral momentum")

# Count overbought/oversold days
overbought = (data['RSI'] > 70).sum()
oversold = (data['RSI'] < 30).sum()
print(f"\n   Overbought days: {overbought}")
print(f"   Oversold days: {oversold}")

### 3.3 MACD (Moving Average Convergence Divergence)

MACD shows the relationship between two moving averages and indicates momentum.

In [ ]:
# Calculate MACD
macd_data = fe.calculate_macd(data)
data['MACD'] = macd_data['macd']
data['MACD_Signal'] = macd_data['signal']
data['MACD_Histogram'] = macd_data['histogram']

print("✅ MACD calculated!")

# Visualize
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Price chart
ax1.plot(data.index, data['Close'], linewidth=2, label='Close Price')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title(f'{ticker} Price and MACD', fontsize=16, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# MACD chart
ax2.plot(data.index, data['MACD'], linewidth=2, label='MACD', color='blue')
ax2.plot(data.index, data['MACD_Signal'], linewidth=2, label='Signal', color='red')
ax2.bar(data.index, data['MACD_Histogram'], label='Histogram', alpha=0.3, color='gray')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.set_ylabel('MACD', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("   - MACD > Signal: Bullish (upward momentum)")
print("   - MACD < Signal: Bearish (downward momentum)")
print("   - Histogram shows the difference between MACD and Signal")

### 3.4 Bollinger Bands

Bollinger Bands show volatility and potential price boundaries.

In [ ]:
# Calculate Bollinger Bands
bb = fe.calculate_bollinger_bands(data, window=20, num_std=2)
data['BB_Middle'] = bb['middle']
data['BB_Upper'] = bb['upper']
data['BB_Lower'] = bb['lower']

print("✅ Bollinger Bands calculated!")

# Visualize
plt.figure(figsize=(15, 6))
plt.plot(data.index, data['Close'], linewidth=2, label='Close Price', color='black')
plt.plot(data.index, data['BB_Middle'], linewidth=2, label='Middle Band (MA 20)', 
         linestyle='--', color='blue')
plt.plot(data.index, data['BB_Upper'], linewidth=2, label='Upper Band', color='red')
plt.plot(data.index, data['BB_Lower'], linewidth=2, label='Lower Band', color='green')
plt.fill_between(data.index, data['BB_Lower'], data['BB_Upper'], alpha=0.1, color='gray')
plt.title(f'{ticker} with Bollinger Bands', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("   - Price near upper band: Potentially overbought")
print("   - Price near lower band: Potentially oversold")
print("   - Bands widen during high volatility")
print("   - Bands narrow during low volatility")

### 3.5 Volatility

Volatility measures how much the price fluctuates.

In [ ]:
# Calculate volatility
data['Volatility_10'] = fe.calculate_volatility(data, window=10)
data['Volatility_30'] = fe.calculate_volatility(data, window=30)

print("✅ Volatility calculated!")

# Visualize
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

# Price chart
ax1.plot(data.index, data['Close'], linewidth=2, label='Close Price')
ax1.set_ylabel('Price ($)', fontsize=12)
ax1.set_title(f'{ticker} Price and Volatility', fontsize=16, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Volatility chart
ax2.plot(data.index, data['Volatility_10'], linewidth=2, label='10-day Volatility')
ax2.plot(data.index, data['Volatility_30'], linewidth=2, label='30-day Volatility')
ax2.set_ylabel('Volatility', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("   - Higher volatility = More risk and opportunity")
print("   - Lower volatility = More stable prices")
print("   - Spikes often occur during market events")

## 4. Quick Feature Engineering: Add All Features at Once

Instead of adding features one by one, use the `add_all_features()` function!

In [ ]:
# Start fresh
print("Loading fresh stock data...")
data_fresh = yf.download(ticker, start="2019-01-01", end="2024-01-01")

print(f"\nOriginal features: {list(data_fresh.columns)}")
print(f"Total columns: {len(data_fresh.columns)}")

# Add all features at once
print("\n" + "="*60)
data_featured = fe.add_all_features(data_fresh, windows=[10, 20, 50])
print("="*60)

print(f"\nNew feature count: {len(data_featured.columns)}")
print(f"Features added: {len(data_featured.columns) - len(data_fresh.columns)}")

# Display all columns
print("\nAll available features:")
for i, col in enumerate(data_featured.columns, 1):
    print(f"  {i:2d}. {col}")

In [ ]:
# Check for missing values
print("Missing values per feature:")
missing = data_featured.isnull().sum()
missing_features = missing[missing > 0].sort_values(ascending=False)

if len(missing_features) > 0:
    print(missing_features)
    print(f"\n⚠️  Some features have NaN values (this is normal for early rows)")
    print(f"   We'll handle this when preparing for modeling.")
else:
    print("✅ No missing values!")

## 5. Feature Correlation Analysis

Let's see which features are most correlated with price.

In [ ]:
# Calculate correlation with Close price
correlations = data_featured.corrwith(data_featured['Close']).sort_values(ascending=False)

print("Top 15 features correlated with Close price:")
print(correlations.head(15))

# Visualize
plt.figure(figsize=(12, 8))
correlations.head(15).plot(kind='barh', color='steelblue')
plt.title('Top 15 Features Correlated with Close Price', fontsize=16, fontweight='bold')
plt.xlabel('Correlation', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for selected features
selected_features = ['Close', 'MA_10', 'MA_20', 'MA_50', 'RSI', 'MACD', 
                    'BB_Upper', 'BB_Lower', 'Volatility_10', 'Volume']

correlation_matrix = data_featured[selected_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.2f')
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 Interpretation:")
print("   - Values close to 1: Strong positive correlation")
print("   - Values close to -1: Strong negative correlation")
print("   - Values close to 0: No correlation")

## 6. Prepare Data for Machine Learning

Now let's prepare our featured data for regression modeling.

In [ ]:
# Prepare regression data
print("Preparing data for machine learning...\n")
X, y = fe.prepare_regression_data(data_featured, target_column='Close', drop_na=True)

print(f"\n✅ Data prepared!")
print(f"   Features (X) shape: {X.shape}")
print(f"   Target (y) shape: {y.shape}")
print(f"   Total samples: {len(X)}")

print(f"\nFeature columns ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nTarget variable: Tomorrow's closing price")
print(f"   Min: ${y.min():.2f}")
print(f"   Max: ${y.max():.2f}")
print(f"   Mean: ${y.mean():.2f}")

## 7. Train/Test Split for Time Series

**IMPORTANT:** For time series, we don't shuffle! Train on old data, test on recent data.

In [ ]:
# Split data (80% train, 20% test)
print("Splitting data for time series...\n")
X_train, X_test, y_train, y_test = fe.time_series_split(X, y, train_size=0.8)

print(f"\n✅ Data split complete!")
print(f"\nTraining set:")
print(f"   Shape: {X_train.shape}")
print(f"   Samples: {len(X_train)}")
print(f"   Date range: {X_train.index[0].date()} to {X_train.index[-1].date()}")

print(f"\nTest set:")
print(f"   Shape: {X_test.shape}")
print(f"   Samples: {len(X_test)}")
print(f"   Date range: {X_test.index[0].date()} to {X_test.index[-1].date()}")

print(f"\n⚠️  Notice: No shuffling! We train on past data and test on recent data.")

## 8. Summary and Next Steps

Congratulations! You've learned how to engineer features for stock prediction.

In [ ]:
print("="*70)
print("FEATURE ENGINEERING SUMMARY")
print("="*70)

print(f"\n📊 Stock analyzed: {ticker}")
print(f"📅 Date range: {data_featured.index[0].date()} to {data_featured.index[-1].date()}")
print(f"📈 Total trading days: {len(data_featured)}")

print(f"\n🔧 Features created: {len(data_featured.columns) - 6}")
print(f"   Original OHLCV columns: 6")
print(f"   Total columns now: {len(data_featured.columns)}")

print(f"\n🎯 Machine Learning Ready:")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")
print(f"   Features per sample: {X_train.shape[1]}")

print(f"\n✅ Technical Indicators Added:")
indicators = [
    "Moving Averages (10, 20, 50)",
    "RSI (Relative Strength Index)",
    "MACD with Signal and Histogram",
    "Bollinger Bands (Upper, Middle, Lower)",
    "Volatility (10 and 30 day)",
    "Daily Returns",
    "Rate of Change (10 day)",
    "Momentum (10 day)",
    "Volume Moving Average (10 day)",
    "Price Position (20 day)"
]
for i, indicator in enumerate(indicators, 1):
    print(f"   {i:2d}. {indicator}")

print(f"\n🚀 Next Steps:")
print(f"   1. Train regression models (Linear, Random Forest, Gradient Boosting)")
print(f"   2. Make predictions on test set")
print(f"   3. Evaluate with RMSE, R², MAE")
print(f"   4. Predict tomorrow's price!")

print(f"\n📚 See these notebooks:")
print(f"   - stock-predictor-COMPLETE.ipynb (Full implementation)")
print(f"   - house-price-predictor.ipynb (Another regression example)")
print(f"   - regression-metrics-complete.ipynb (Evaluation methods)")

print("\n" + "="*70)
print("READY TO BUILD THE STOCK PREDICTOR! 🎉")
print("="*70)

## Quick Reference: Feature Engineering Functions

```python
# Import the library
import feature_engineering as fe

# Individual indicators
fe.calculate_moving_average(data, window=10)
fe.calculate_rsi(data, window=14)
fe.calculate_macd(data)
fe.calculate_bollinger_bands(data, window=20)
fe.calculate_volatility(data, window=10)
fe.calculate_daily_returns(data)
fe.calculate_rate_of_change(data, window=10)
fe.calculate_momentum(data, window=10)

# Quick setup (recommended!)
data = fe.add_all_features(data, windows=[10, 20, 50])

# Prepare for ML
X, y = fe.prepare_regression_data(data)
X_train, X_test, y_train, y_test = fe.time_series_split(X, y)
```

## Indicator Reference

| Indicator | What It Measures | Typical Use |
|-----------|-----------------|-------------|
| **MA** | Price trend | Identify overall direction |
| **RSI** | Momentum (0-100) | Overbought/oversold signals |
| **MACD** | Trend momentum | Buy/sell signals |
| **Bollinger Bands** | Volatility | Price boundaries |
| **Volatility** | Price fluctuation | Risk assessment |
| **Returns** | Daily price change | Performance tracking |
| **ROC** | Rate of change | Momentum strength |
| **Momentum** | Price movement | Trend strength |
| **Volume MA** | Trading activity | Liquidity assessment |
| **Price Position** | Position in range | Support/resistance |

---

**Created:** 2026-01-05  
**Module:** 3 - Regression  
**Course:** ML for Engineers  
**Next:** Build the stock predictor!